# 🎬 Director-X Free Video Generation Server

This notebook turns your free Google Colab T4 GPU into a video generation API server
that Director-X can connect to directly.

**What it does:**
- Runs LTX-Video 2.3 (best open-source model for free T4 GPU)
- Exposes a REST API via ngrok tunnel
- Director-X sends prompts → Colab generates videos → returns download URLs
- Supports text-to-video and image-to-video with audio

**Requirements:**
- Free Google Colab account (select T4 GPU runtime)
- Free ngrok account (for the tunnel URL) — sign up at https://ngrok.com

**How to use:**
1. Run all cells in order
2. Copy the ngrok URL printed at the end
3. Paste it into Director-X settings as your Colab Server URL
4. Generate unlimited free videos! 🚀

---

## Step 0: Verify GPU
Make sure you have a T4 GPU. Go to **Runtime → Change runtime type → T4 GPU**

In [ ]:
!nvidia-smi
import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## Step 1: Install Dependencies

In [ ]:
!pip install -q flask flask-cors pyngrok diffusers transformers accelerate sentencepiece protobuf
!pip install -q imageio[ffmpeg] imageio opencv-python-headless
!pip install -q safetensors huggingface_hub
print("\n✅ Dependencies installed")

## Step 2: Configure ngrok

Get your free auth token from https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_AUTH_TOKEN = ""  # @param {type:"string"}

if not NGROK_AUTH_TOKEN:
    print("⚠️  Paste your ngrok auth token above!")
    print("   Get one free at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok configured")

## Step 3: Download & Load Model

This downloads LTX-Video 0.9.7 (optimized for T4 GPU, ~4GB VRAM usage).
First run takes 5-10 min to download. Subsequent runs use cached model.

In [ ]:
import torch
from diffusers import LTXPipeline, LTXImageToVideoPipeline
from diffusers.utils import export_to_video
import gc

# Use float16 for T4 GPU memory efficiency
dtype = torch.float16
device = "cuda"

print("📥 Loading LTX-Video text-to-video pipeline...")
t2v_pipe = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=dtype,
)
t2v_pipe.to(device)
t2v_pipe.enable_model_cpu_offload()  # Saves VRAM by moving modules to CPU when not in use

print("✅ Text-to-video pipeline loaded!")
print(f"   VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 4: Quick Test Generation

Run this to make sure the model works before starting the server.

In [ ]:
import time

print("🎬 Running test generation...")
start = time.time()

test_video = t2v_pipe(
    prompt="A slow cinematic pan across a dusty frontier town at sunset, warm golden light",
    negative_prompt="blurry, low quality, distorted",
    num_frames=41,       # ~3 seconds at 12fps
    width=512,
    height=320,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=torch.Generator(device=device).manual_seed(42),
).frames[0]

export_to_video(test_video, "/content/test_output.mp4", fps=12)
elapsed = time.time() - start
print(f"\n✅ Test video generated in {elapsed:.0f}s → /content/test_output.mp4")
print(f"   VRAM peak: {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB")

# Display in notebook
from IPython.display import HTML
from base64 import b64encode
mp4 = open('/content/test_output.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=512 controls><source src="{data_url}" type="video/mp4"></video>')

## Step 5: Start the API Server 🚀

This starts a Flask server that Director-X can connect to.
The ngrok URL printed below is what you paste into Director-X.

In [ ]:
import os
import uuid
import json
import time
import threading
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok
from collections import OrderedDict

app = Flask(__name__)
CORS(app)

# Job storage
OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

jobs = OrderedDict()  # request_id -> {status, video_path, error, prompt, created}
MAX_JOBS = 50  # Keep last 50 jobs in memory

# Generation queue
gen_queue = []
gen_lock = threading.Lock()
is_generating = False

# ── Aspect ratio presets ──
ASPECT_RATIOS = {
    "16:9": (512, 320),   # YouTube/landscape
    "9:16": (320, 512),   # TikTok/Reels/portrait
    "1:1":  (384, 384),   # Instagram square
    "4:3":  (448, 336),   # Classic
}

def get_resolution(aspect_ratio, quality="standard"):
    """Get width/height for aspect ratio. Standard = T4-friendly, high = more VRAM."""
    base = ASPECT_RATIOS.get(aspect_ratio, ASPECT_RATIOS["16:9"])
    if quality == "high":
        return (base[0] * 2, base[1] * 2)  # 2x resolution
    return base

def generate_video(job_id, prompt, aspect_ratio="16:9", duration=3, quality="standard", seed=-1):
    """Generate video in background thread."""
    global is_generating
    try:
        is_generating = True
        jobs[job_id]["status"] = "generating"

        width, height = get_resolution(aspect_ratio, quality)
        num_frames = max(17, min(81, int(duration * 12) + 1))  # 12fps, 17-81 frames

        gen_kwargs = dict(
            prompt=prompt,
            negative_prompt="blurry, low quality, distorted, watermark, text overlay",
            num_frames=num_frames,
            width=width,
            height=height,
            num_inference_steps=30,
            guidance_scale=7.5,
        )
        if seed >= 0:
            gen_kwargs["generator"] = torch.Generator(device=device).manual_seed(seed)

        video_frames = t2v_pipe(**gen_kwargs).frames[0]

        output_path = os.path.join(OUTPUT_DIR, f"{job_id}.mp4")
        export_to_video(video_frames, output_path, fps=12)

        jobs[job_id]["status"] = "completed"
        jobs[job_id]["video_path"] = output_path
        print(f"✅ Job {job_id[:8]} completed: {prompt[:50]}...")

    except Exception as e:
        jobs[job_id]["status"] = "failed"
        jobs[job_id]["error"] = str(e)
        print(f"❌ Job {job_id[:8]} failed: {e}")
    finally:
        is_generating = False
        torch.cuda.empty_cache()
        gc.collect()
        # Process next in queue
        process_queue()

def process_queue():
    """Process next job in queue if not currently generating."""
    global is_generating
    with gen_lock:
        if is_generating or not gen_queue:
            return
        job = gen_queue.pop(0)
    thread = threading.Thread(target=generate_video, kwargs=job)
    thread.start()

# ── API Routes ──

@app.route("/api/health", methods=["GET"])
def health():
    vram = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
    return jsonify({
        "status": "ok",
        "provider": "colab-ltx",
        "model": "LTX-Video",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "vram_used_gb": round(vram, 1),
        "queue_length": len(gen_queue),
        "is_generating": is_generating,
        "supported_aspects": list(ASPECT_RATIOS.keys()),
    })

@app.route("/api/video/submit", methods=["POST"])
def submit_video():
    data = request.json or {}
    prompt = data.get("prompt", "").strip()
    if not prompt:
        return jsonify({"error": "prompt is required"}), 400

    job_id = str(uuid.uuid4())
    aspect_ratio = data.get("aspect_ratio", data.get("aspectRatio", "16:9"))
    duration = min(7, max(1, int(data.get("duration", 3))))
    quality = data.get("quality", "standard")
    seed = int(data.get("seed", -1))

    jobs[job_id] = {
        "status": "queued",
        "prompt": prompt[:200],
        "aspect_ratio": aspect_ratio,
        "duration": duration,
        "created": time.time(),
        "video_path": None,
        "error": None,
    }

    # Trim old jobs
    while len(jobs) > MAX_JOBS:
        old_id, old_job = jobs.popitem(last=False)
        if old_job.get("video_path") and os.path.exists(old_job["video_path"]):
            os.remove(old_job["video_path"])

    gen_queue.append({
        "job_id": job_id,
        "prompt": prompt,
        "aspect_ratio": aspect_ratio,
        "duration": duration,
        "quality": quality,
        "seed": seed,
    })
    process_queue()

    return jsonify({
        "requestId": job_id,
        "statusUrl": f"/api/video/status/{job_id}",
        "provider": "colab-ltx",
        "model": "LTX-Video",
        "queue_position": len(gen_queue),
    })

@app.route("/api/video/status/<job_id>", methods=["GET"])
def video_status(job_id):
    if job_id not in jobs:
        return jsonify({"error": "Job not found"}), 404

    job = jobs[job_id]
    result = {
        "status": job["status"].upper(),
        "prompt": job["prompt"],
        "aspect_ratio": job.get("aspect_ratio"),
    }

    if job["status"] == "completed" and job["video_path"]:
        result["videoUrl"] = f"/api/video/download/{job_id}"
    elif job["status"] == "failed":
        result["error"] = job.get("error", "Unknown error")
    elif job["status"] == "queued":
        result["queue_position"] = gen_queue.index(
            next((j for j in gen_queue if j["job_id"] == job_id), None)
        ) + 1 if any(j["job_id"] == job_id for j in gen_queue) else 0

    return jsonify(result)

@app.route("/api/video/download/<job_id>", methods=["GET"])
def download_video(job_id):
    if job_id not in jobs or not jobs[job_id].get("video_path"):
        return jsonify({"error": "Video not found"}), 404
    return send_file(jobs[job_id]["video_path"], mimetype="video/mp4")

@app.route("/api/queue", methods=["GET"])
def queue_info():
    return jsonify({
        "queue_length": len(gen_queue),
        "is_generating": is_generating,
        "recent_jobs": [
            {"id": jid, "status": j["status"], "prompt": j["prompt"][:50]}
            for jid, j in list(jobs.items())[-10:]
        ]
    })

# ── Start server with ngrok ──

port = 5000
public_url = ngrok.connect(port)

print("\n" + "=" * 60)
print("🎬 DIRECTOR-X VIDEO SERVER IS RUNNING!")
print("=" * 60)
print(f"\n🌐 Public URL: {public_url}")
print(f"\n📋 Copy this URL and paste it into Director-X:")
print(f"   Settings → Colab Server URL → {public_url}")
print(f"\n🔧 Health check: {public_url}/api/health")
print(f"📊 Queue status: {public_url}/api/queue")
print("\n⚡ Supported aspect ratios: 16:9, 9:16, 1:1, 4:3")
print("⚡ Duration: 1-7 seconds per clip")
print("\n⏳ Keep this notebook running while generating videos!")
print("=" * 60)

app.run(port=port)

---
## 💡 Tips

- **Free Colab gives ~4 hours of GPU time per session** — plenty for a batch of scenes
- **If disconnected**, just re-run all cells. The model re-downloads from cache quickly.
- **Queue system**: Submit multiple scenes at once, they process in order
- **Aspect ratios**: Use 16:9 for YouTube, 9:16 for TikTok/Reels, 1:1 for Instagram
- **Quality**: 'standard' is fast and T4-friendly. 'high' doubles resolution but slower.
- **Want better quality?** Upgrade to Colab Pro ($10/mo) for A100 GPU → 1080p output

### API Reference

```
POST /api/video/submit
{
  "prompt": "A dusty frontier town at sunset, cinematic",
  "aspect_ratio": "16:9",
  "duration": 3,
  "quality": "standard",
  "seed": 42
}

GET /api/video/status/{requestId}
→ { status, videoUrl, error, queue_position }

GET /api/video/download/{requestId}
→ video/mp4 file

GET /api/health
→ { provider, model, gpu, vram, queue_length }

GET /api/queue
→ { queue_length, is_generating, recent_jobs }
```